In [3]:
import sys
import numpy as np
import importlib

sys.path.append('../src')
import policies 
import bbDebiasing2


Sanity check: if all my initial policies have single LS, should end up w predictor of mean.

In [58]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[1,0],[1,0]])] #debiasing wrt these LS will just give you average predictor
train_ys = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.1

bbModeltest = bbDebiasing2.bbModel(my_policy, other_policies,train_ys, curr_preds, tolerance)
print(bbModeltest.debias()==np.tile(train_ys.mean(axis=0), (len(train_ys),1)))
bbModeltest.predict(curr_preds, other_policies)==np.tile(train_ys.mean(axis=0), (len(train_ys),1))

[[ True  True]
 [ True  True]
 [ True  True]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

Sanity check: if initial policies have $n$ disjoint level sets, should converge towards exact labels. Note: they don't go exactly to this because the maximal level set in each round often is the one where the policy is 0 in multiple coordinates, so do end up iterating through LSs over many rounds until tolerance constraint is met. 

In [57]:
importlib.reload(bbDebiasing2)
curr_preds = np.zeros([3,2])
my_policy = policies.Simplex(2)
other_policies = [np.array([[1,0],[0,1],[0,0]])] #debiasing wrt these LS will just give you average predictor
train_y = np.array([[1,0],[2,-1],[3,5]])
tolerance=0.01

bbModeltest2 = bbDebiasing2.bbModel(my_policy, other_policies,train_ys,curr_preds, tolerance)
print(bbModeltest2.debias())
bbModeltest2.predict(curr_preds, other_policies)==bbModeltest2.debias()

[[ 1.0078125   0.01953125]
 [ 2.         -1.        ]
 [ 2.9921875   4.98046875]]


array([[ True,  True],
       [ True,  True],
       [ True,  True]])

Verifying that running bias calculations w different shapes of masks works

In [92]:
def _calculate_bias(input_masks, train_ys, curr_preds):
    """
    For all c in C (masks array), calculates E[y - h(x)|x in c]
    """
    n_samples = len(train_ys)
    # converting masks to floats and expanding dimension so can broadcast
    masks = input_masks.astype(np.float32)
    masks = np.expand_dims(masks, axis=-1)
    diffs = train_ys*masks - curr_preds*masks
    sums = np.sum(diffs, axis=-2)
    ns = np.sum(masks, axis=-2)
    bias = np.where(ns>0, sums/ns.clip(min=1),0) # clip is to avoid div by 0 errors
    probs = np.squeeze(ns/n_samples)
    return bias, probs

In [ ]:
_calculate_bias

First checking with own models' level sets

In [148]:
n_samples = 5
n_coords = 3
n_bins = 2
input_masks = np.random.binomial(1, 1/2, (n_coords, n_bins, n_samples)).astype(bool)
train_ys = np.random.binomial(1,1/2,(n_samples, n_bins))
curr_preds = np.random.binomial(1,1/2,(n_samples, n_bins))

print(train_ys)
print(curr_preds)
print(input_masks)

[[1 1]
 [0 0]
 [0 1]
 [0 1]
 [0 1]]
[[0 0]
 [0 0]
 [1 0]
 [1 0]
 [1 0]]
[[[False  True  True False False]
  [ True  True  True False  True]]

 [[False False  True  True False]
  [False False  True False  True]]

 [[False False False  True  True]
  [False False  True False  True]]]


In [149]:
own_model_bias,own_model_probs = _calculate_bias(input_masks, train_ys, curr_preds)

In [151]:
test_indices = (1,1)
print(np.mean(train_ys[input_masks[test_indices]]-curr_preds[input_masks[test_indices]], axis=0))
print(own_model_bias[test_indices])
print(input_masks[test_indices].sum()/n_samples)
print(own_model_probs[test_indices])

[-1.  1.]
[-1.  1.]
0.4
0.4


Now checking with maximal models' level sets

In [153]:
n_policies = 4
input_masks = np.random.binomial(1, 1/2, (n_policies, n_samples)).astype(bool)
max_model_bias,max_model_probs = _calculate_bias(input_masks, train_ys, curr_preds)

In [154]:
test_indices = -1
print(np.mean(train_ys[input_masks[test_indices]]-curr_preds[input_masks[test_indices]], axis=0))
print(max_model_bias[test_indices])
print(input_masks[test_indices].sum()/n_samples)
print(max_model_probs[test_indices])


[-1.  1.]
[-1.  1.]
0.2
0.2


Can I get the find max bias code to work for both datatypes?

In [156]:
def _find_maximum_bias(bias, probs):
    l_infinity = np.max(np.abs(bias), axis=-1)
    weighted_bias = probs*l_infinity
    max_weighted_bias = weighted_bias.max()
    target_set = tuple(np.argwhere(weighted_bias==max_weighted_bias)[0]) #converting into tuple for indexing
    target_bias = bias[target_set]
    return max_weighted_bias, target_set, target_bias
    return None

In [162]:
_find_maximum_bias(own_model_bias, own_model_probs)

(0.6000000089406967, (0, 1), array([-0.25,  0.75]))

In [171]:
np.max(np.abs(own_model_bias), axis=-1)*own_model_probs

array([[0.2       , 0.60000001],
       [0.40000001, 0.40000001],
       [0.40000001, 0.40000001]])

In [169]:
np.max(np.abs(max_model_bias), axis=-1)*max_model_probs

array([0.40000001, 0.        , 0.40000002, 0.2       ])

In [170]:
_find_maximum_bias(max_model_bias, max_model_probs)

(0.4000000158945719, (2,), array([-0.66666667,  0.66666667]))

Trying to fix the masking to be the proper dimension since need to be jointly calibrated over product of the two kinds of sets.

In [174]:
foo = np.random.binomial(1,0.5,6).astype(bool)
bar = np.random.binomial(1,0.5,(2,6)).astype(bool)

print(foo)
print(bar)
print(foo*bar)

[False  True  True False False False]
[[False  True  True  True  True False]
 [ True  True  True False False  True]]
[[False  True  True False False False]
 [False  True  True False False False]]


In [32]:
max_model_masks = np.random.binomial(1,0.5,(2,5)).astype(bool)
model_ls_masks = np.random.binomial(1,0.5,(2,3,4,5)).astype(bool)
max_model_masks_reshaped = max_model_masks[:,None,None,None,:]
out = model_ls_masks * max_model_masks_reshaped

In [67]:
import itertools 

out = np.zeros((2,3,4,2,5))
for (k,m,d) in itertools.product(np.arange(2), np.arange(3), np.arange(4)):
    out[k,m,d] = model_ls_masks[k,m,d]*max_model_masks

In [65]:
out2 = np.repeat(model_ls_masks[:, :, :, np.newaxis, :], 2, axis=3)*max_model_masks


In [73]:
out3 = np.repeat(np.expand_dims(model_ls_masks,-2),2,axis=3)*max_model_masks

In [75]:
for (k,m,d,i,j) in itertools.product(np.arange(2), np.arange(3), np.arange(4), np.arange(2), np.arange(5)):
    if out[k,m,d,i,j]!=out3[k,m,d,i,j]:
        print('aah')

Getting the bias calculations to be over all of the models predictions.

In [68]:
k,d,m,n = 2,3,4,5
masks = np.random.binomial(1,0.6,(k,d,m,k,n))
preds_by_models = np.random.binomial(1,0.6,(k,n,d))
train_ys = np.random.binomial(1,0.6,(n,d))

In [69]:
masks = np.expand_dims(masks, axis=-1)
diffs = masks*train_ys - masks*preds_by_models
sums = np.sum(diffs, axis=-2)
ns = np.sum(masks, axis=-2)
bias = np.where(ns>0, sums/ns.clip(min=1), 0) 
probs = np.squeeze(ns/n)

In [71]:
l_infinity = np.max(np.abs(bias), axis=-1) # shape k x d x m x k
weighted_bias = probs*l_infinity
max_weighted_bias = weighted_bias.max()
target_set = tuple(np.argwhere(weighted_bias==max_weighted_bias)[0]) #converting into tuple for indexing
target_bias = bias[target_set]